Research Question: Why are Claudin 10b and Claudin 16/19 seperated in the kidney (they have very different permeabilities for Na+ and Mg2+)?

In [ ]:
import matplotlib.pyplot as plt

from paracellular_transport import (
    TAL_STATE1,
    TAL_STATE2_AVG,
    TAL_STATE2_PARALLEL,
    plot_flow_dynamics,
    plot_ion_dynamics,
    plot_membrane_potential,
    plot_total_flux_comparison,
    run_scenario,
    transepithelial_potential,
)

Using **Nernst-Equation** the EMF for the individual ions at each boundary is calculated

*(implemented in `paracellular_transport.physics.nernst_equation`)*

Using **Goldman-Equation** the equilibrium potential is calculated for each boundary:

*(implemented in `paracellular_transport.physics.goldmann_equation`)*

Extend Goldman-Equation to divalent ions

*(implemented in `paracellular_transport.physics.divalent_membrane_potential`; the electromotive-force helper is `paracellular_transport.physics.EMF`)*

Flux is proportional to EMF, permeability and the concentration in the donor compartment

*(implemented in `paracellular_transport.physics.calculate_flux`)*

Using Flux, the **new number of ions** is calculated

*(implemented in `paracellular_transport.physics.calculate_ion_change`)*

For parallel connected tight junctions that have different potentials, we have to calculate a shared voltage U_k

*(implemented in `paracellular_transport.physics.shared_voltage`; `Compartment`/`Junctions` live in `paracellular_transport.models`)*

All scenario parameters (concentrations, permeabilities, resistances, voltage clamps, timestep, divalent mode) now live in `paracellular_transport.config` as `Scenario` objects -- `TAL_STATE1`, `TAL_STATE2_PARALLEL`, `TAL_STATE2_AVG`. Adding a new nephron segment (e.g. Proximal Tubule) means adding a new `Scenario` there; the simulation loop below does not change.

In [ ]:
result_state1 = run_scenario(TAL_STATE1)
result_state2_parallel = run_scenario(TAL_STATE2_PARALLEL)
result_state2_avg = run_scenario(TAL_STATE2_AVG)

In [ ]:
# Total transepithelial potentials (sum of every junction's potential in a pathway's chain)
transepithelial_potential_state1 = transepithelial_potential(result_state1.potential_history['10b'])
transepithelial_potential_10b = transepithelial_potential(result_state2_parallel.potential_history['10b'])
transepithelial_potential_16_19 = transepithelial_potential(result_state2_parallel.potential_history['16_19'])
transepithelial_potential_avg = transepithelial_potential(result_state2_avg.potential_history['avg'])

In [ ]:
# State 1 (mTAL)
plot_flow_dynamics(result_state1.flow_history['10b'], result_state1.time_axis, title="Flow Dynamics: State 1 Cldn10b")
plot_membrane_potential(transepithelial_potential_state1, result_state1.time_axis, title="Potential: State 1")
plot_ion_dynamics(result_state1.concentration_history, result_state1.time_axis)

In [ ]:
# State 2, parallel claudins (cTAL)
plot_flow_dynamics(result_state2_parallel.flow_history['10b'], result_state2_parallel.time_axis, title="Flow Dynamics: Cldn10b (parallel)")
plot_flow_dynamics(result_state2_parallel.flow_history['16_19'], result_state2_parallel.time_axis, title="Flow Dynamics: Cldn16/19 (parallel)")
plot_membrane_potential(transepithelial_potential_10b, result_state2_parallel.time_axis, title="Potential: Cldn10b")
plot_membrane_potential(transepithelial_potential_16_19, result_state2_parallel.time_axis, title="Potential: Cldn16/19")
plot_ion_dynamics(result_state2_parallel.concentration_history, result_state2_parallel.time_axis)

In [ ]:
# State 2, averaged claudin (cTAL)
plot_flow_dynamics(result_state2_avg.flow_history['avg'], result_state2_avg.time_axis, title="Flow Dynamics: Averaged * 2")
plot_membrane_potential(transepithelial_potential_avg, result_state2_avg.time_axis, title="Potential: Averaged")
plot_ion_dynamics(result_state2_avg.concentration_history, result_state2_avg.time_axis)

Research Question: Is the sum of the ion fluxes through the combined system larger than 2 times the flux through the AVG system (assuming that half of the junctions are 10b, half are 16/19)? Checked for Na+, Cl-, and Mg2+.

In [ ]:
plot_total_flux_comparison(result_state2_parallel, result_state2_avg, ion='Mg')
# plot_total_flux_comparison(result_state2_parallel, result_state2_avg, ion='Na')
# plot_total_flux_comparison(result_state2_parallel, result_state2_avg, ion='Cl')

to check what causes the reabsorption difference - because equal total permeabilities makes it very confusing -, check sodium flux, potential values, test what happens if 10b and 1619 where the same as avg